In [78]:
# import libraries
import os
import json
import uuid
import datetime
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [79]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment variables
open_ai_key = os.getenv("SBS_OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()
client.api_key = open_ai_key

JSONL_FILE = "requests.jsonl"
OUTPUT_FILE = "output.jsonl"
VOCAB_FOLDER_PATH = "pali_class/vocab"
EXERCISE_FOLDER_PATH = "pali_class/exercises"

In [80]:
keyword = input("Enter a Pali word: ").strip() # Remove extra spaces

def search_pali_in_csv(keyword):
    """Search for a Pali word in all CSV files in the folder."""
    result = {
        "id": -1,
        "pali": "",
        "pos": "",
        "exercise_number": ""
    }

    for filename in os.listdir(VOCAB_FOLDER_PATH):
        if filename.endswith(".csv"): # Only search in CSV files
            file_path = os.path.join(VOCAB_FOLDER_PATH, filename)
            df = pd.read_csv(file_path, dtype=str) 
            match = df[df["pali"] == keyword]

            if not match.empty:
                result["id"] = match["id"].values[0]
                result["pali"] = match["pali"].values[0]
                result["pos"] = match["pos"].values[0]
                
                print("Match found in file:", filename)
                
                number = filename.split("_")[-1].split(".")[0] # Extract the number
                result["exercise_number"] = number

                break

    if result["id"] == -1:
        print("No matches found")

    return result

result = search_pali_in_csv(keyword)
result

Match found in file: vocab_class_2.csv


{'id': '2603', 'pali': 'attha 2.1', 'pos': 'masc', 'exercise_number': '2'}

In [81]:
# Find exercise number
exercise_number = result['exercise_number']
found_exercise = False
target_exercise = ""
exercise_data = ""

for filename in os.listdir(EXERCISE_FOLDER_PATH):
    if filename.endswith(".txt") and f"_{exercise_number}." in filename:
        found_exercise = True
        target_exercise = filename
        print("Found:", filename)

if not found_exercise:
    print("Exercise not found")
else:
    exercise_data = open(os.path.join(EXERCISE_FOLDER_PATH, target_exercise), "r").read().strip()
    print(exercise_data)

Found: exercises_class_2.txt
Class 2 Exercises

namo tassa bhagavato arahato sammā-sambuddhassa	 					
Homage to him, the Blessed One, the Worthy One, the fully Enlightened One.
 
avijjāya tv'eva asesa-virāga-nirodhā saṅkhāra-nirodho, saṅkhāra-nirodhā viññāṇa-nirodho, viññāṇa-nirodhā nāmarūpa-nirodho, nāmarūpa-nirodhā saḷāyatana-nirodho, saḷāyatana-nirodhā phassa-nirodho, phassa-nirodhā vedanā-nirodho, vedanā-nirodhā taṇhā-nirodho, taṇhā-nirodhā upādāna-nirodho, upādāna-nirodhā bhava-nirodho, bhava-nirodhā jāti-nirodho, jāti-nirodhā jarā-maraṇaṃ soka-parideva-dukkha-domanass'upāyāsā nirujjhanti. 
But from the complete fading away and cessation of ignorance there is cessation of volitional formations; from the cessation of volitional formations, cessation of consciousness; from the cessation of consciousness, cessation of name-and-form; from the cessation of name-and-form, cessation of the six sense bases; from the cessation of the six sense bases, cessation of contact; from the cessati

In [82]:
SYSTEM_PROMPT = """
You are an AI assistant specialized in Pali language processing. Your task is to extract relevant Pali sentences based on the given keywords, ensuring grammatical accuracy and meaningful context.

For each keyword:
1. Extract exactly **one** relevant Pali sentence.
2. If multiple sentences exist, **prioritize** those without special markers ('$' or '%'). If unavoidable, use the best option available.
3. Preserve existing 'simpl' markers.
4. Provide the **English translation** of the sentence.
5. Include the **source reference**.
6. If no suitable sentence is found, return a flag for manual review.

### Output Format (JSON):
{
  "pali_sentence": "...",
  "translation": "...",
  "source": "...",
  "flag_review": false
}
If no match is found:
{
  "pali_sentence": null,
  "translation": null,
  "source": null,
  "flag_review": true
}

Ensure the response is structured, accurate, and follows these rules.
"""

In [84]:
USER_PROMPT = f"""
For the Pali word "{result['pali']}", which has "{result['pos']}" as its grammatical part of speech, 
find example sentences in "{exercise_data}".

### **Output format (tab-separated):**  
Pali Word "tab" Source (e.g., MN12, SN12.3, from exercises_class_3.txt) "tab"  
Sutta (Name of the Source) "tab" Example Pali (bold the Pali word as <b>{result['pali']}</b>) "tab"
English Translation (copy from exercises_class_3.txt, no need to generate your own)

Ensure that the **Pali word is bolded** in the example Pali sentence.
"""

def generate_unique_id(user_id):
    timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S") # Format: YYYYMMDDHHMMSS
    unique_str = uuid.uuid4().hex
    return f"{user_id}-{timestamp}-{unique_str}"

# Generate a unique ID for the request (user-timestamp-unique)
generate_unique_id = generate_unique_id("user-1")

request = {
    "custom_id": generate_unique_id, 
    "method": "POST", 
    "url": "/v1/chat/completions", 
    "body": {
        "model": "gpt-3.5-turbo-0125", 
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],
        "max_tokens": 1000
    }
}

# Write to JSONL file (append mode for multiple requests)
with open(JSONL_FILE, "a", encoding="utf-8") as f:
    json.dump(request, f) # Convert dictionary to JSON string
    f.write("\n") # Newline for the next JSON object

print(f"Request saved to {JSONL_FILE}")

Request saved to requests.jsonl


In [99]:
# Define single request
response = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=[
        {
            "role": "system", 
            "content": "You are helpful assistant."
        },
        {
            "role": "user", 
            "content": "I love your help."}
    ],
    max_tokens=1000
)

# Print response
print(response)

ChatCompletion(id='chatcmpl-B3GDjbkRX00nDuqx58Yflvs1EiRn7', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Thank you! I'm glad to be of assistance. If you have any questions or need help with anything, feel free to ask!", role='assistant', function_call=None, tool_calls=None, refusal=None))], created=1740117035, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=28, prompt_tokens=21, total_tokens=49, prompt_tokens_details={'cached_tokens': 0, 'audio_tokens': 0}, completion_tokens_details={'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}))


In [85]:
batch_input_file = client.files.create(
    file=open(JSONL_FILE, "rb"),
    purpose="batch"
)

batch_input_file

FileObject(id='file-LAEqKjyGAPTrEXGxArwc2n', bytes=15116, created_at=1740116690, filename='requests.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)

In [86]:
batch_input_file_id = batch_input_file.id

# Start the batch process and store the response
batch_request = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": "Automated batch processing"}
)

# Get Batch ID
batch_id = batch_request.id
print("Batch ID:", batch_id)

Batch ID: batch_67b812d7c18881909b030da0a490c343


In [97]:
# Check batch status
client.batches.retrieve(batch_id)

Batch(id='batch_67b812d7c18881909b030da0a490c343', completion_window='24h', created_at=1740116695, endpoint='/v1/chat/completions', input_file_id='file-LAEqKjyGAPTrEXGxArwc2n', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1740116789, error_file_id=None, errors=None, expired_at=None, expires_at=1740203095, failed_at=None, finalizing_at=1740116788, in_progress_at=1740116696, metadata={'description': 'Automated batch processing'}, output_file_id='file-3FNMaqZDxxS6uMEGkMK8mb', request_counts=BatchRequestCounts(completed=2, failed=0, total=2))

In [98]:
file_response = client.files.content("file-3FNMaqZDxxS6uMEGkMK8mb")
print(file_response.text)

{"id": "batch_req_67b81334c77c8190be644166a8e81b78", "custom_id": "user-1-20250221134424-bc7948efd66b4e0e870a75a64a950ce5", "response": {"status_code": 200, "request_id": "de88c96277332f548f0199f2eea352b5", "body": {"id": "chatcmpl-B3G9gE1c5IOOXWKxC13QIrEZynXCm", "object": "chat.completion", "created": 1740116784, "model": "gpt-3.5-turbo-0125", "choices": [{"index": 0, "message": {"role": "assistant", "content": "```json\n{\n  \"pali_sentence\": \"<b>attha 2.1</b>\",\n  \"translation\": \"But from the complete fading away and cessation of ignorance there is cessation of volitional formations; from the cessation of volitional formations, cessation of consciousness; from the cessation of consciousness, cessation of name-and-form; from the cessation of name-and-form, cessation of the six sense bases; from the cessation of the six sense bases, cessation of contact; from the cessation of contact, cessation of feeling; from the cessation of feeling, cessation of craving; from the cessation o